# A LangGraph-Orchestrated Framework for Secure Analysis and Repair of LLM-Generated Code

## Experimental Evaluation Notebook

**Authors**

Lizeth Campos Velazquez  
Bikash Chandra Singh  

California State University, Fresno

---

This notebook accompanies the paper and provides a reviewer-facing interface for inspecting the frozen workflow configuration, benchmark
materials, experiment scripts, prompts, archived outputs, and reported
evaluation results.

All displayed numerical results are loaded from the
archived evaluation artifacts used to prepare the paper.

## Notebook Contents

1. [Overview](#overview)
2. [Reviewer Checklist](#reviewer-checklist)
3. [Repository Organization](#repository-organization)
4. [Experimental Workflow](#experimental-workflow)
5. [Environment Setup](#environment-setup)
6. [Frozen Configuration](#final-configuration-summary)
7. [Benchmark Inspection](#benchmark-inspection)
8. [Prompts and Experiment Scripts](#prompts-and-scripts)
9. [Reported Calibration Results](#workflow-calibration)
10. [Reported Detection Results](#experiment-1-detection-and-comparative-analysis)
11. [Reported Remediation Results](#experiment-2-end-to-end-secure-remediation)
12. [Reported Chained-Vulnerability Results](#experiment-3-chained-vulnerability-evaluation)
13. [Reported Runtime and Cost](#runtime-and-cost-analysis)
14. [Paper Table Inventory](#paper-table-inventory)
15. [Supporting Logs and Outputs](#supporting-artifacts)
16. [Artifact Summary](#artifact-summary)
17. [References](#references)

<a name="overview"></a>

# 1. Overview

This notebook provides structured access to the public evaluation
repository accompanying the paper.

Its purpose is to help reviewers inspect:

- the frozen workflow configuration;
- benchmark samples and metadata;
- prompts used during evaluation;
- experiment scripts;
- archived experiment logs and outputs;
- Tables I–VIII reported in the paper;
- the architecture figure;
- runtime and cost provenance.



### Scope of This Artifact

The repository contains:

- the 138-case SecurityEval-derived benchmark;
- the 18-case calibration subset;
- chained-vulnerability scenario materials;
- frozen workflow documentation;
- evaluation prompts;
- experiment scripts;
- experiment logs;
- representative model and tool outputs;
- reported-result CSV files;
- the architecture figure;
- runtime and cost documentation.

The experiment scripts document how the evaluation was conducted.
They are included for methodological transparency but are not executed
by this notebook.

<a name="reviewer-checklist"></a>

# 2. Reviewer Checklist

This notebook allows reviewers to:

- [ ] Confirm that the public repository loads successfully
- [ ] Inspect the frozen workflow configuration
- [ ] Inspect the benchmark organization
- [ ] Locate prompts and experiment scripts
- [ ] View reported Tables I–VIII
- [ ] Inspect the architecture figure
- [ ] Locate experiment logs and representative outputs
- [ ] Review runtime and cost provenance
- [ ] Confirm that all paper-supporting artifacts are available

<a name="repository-organization"></a>
# 3. Repository Organization

The evaluation repository is organized as follows:

```text
repository-root/
│
├── configs/
│   ├── README.md
│   └── workflow_configuration.md
│
├── datasets/
│   ├── chains/
│   ├── securityeval/
│   └── README.md
│
├── experiments/
│   ├── README.md
│   ├── experiment1/
│   ├── experiment2/
│   ├── experiment3/
│   ├── calibration/
│   └── methodology.md
│
├── notebooks/
│   ├── README.md
│   └── paper_reproduction.ipynb
│
├── prompts/
│   ├── chain_reasoning_prompt.md
│   ├── chain_repair_prompt.md
│   ├── detector_prompt.md
│   ├── repair_prompt.md
│   ├── secure_counterpart_generation_prompt.md
│   ├── langgraph_end_to_end_workflow_repair_prompt.md
│   └── README.md
│
├── results/
│   ├── paper_results/
│   └── README.md

│
├── CITATION.cff
├── requirements.txt
└── README.md

### Directory Roles

| Directory | Purpose |
|---|---|
| `configs/` | Documents the calibrated and frozen experimental configuration. |
| `datasets/` | Contains or documents the benchmark data used in the evaluation. |
| `experiments/` | Contains experiment scripts, methodology, and evaluation utilities. |
| `notebooks/` | Provides the reviewer-facing inspection notebook. |
| `prompts/` | Stores the prompt templates used during benchmark preparation and experimentation. |
| `results/paper_results/` | Stores the archived outputs used to produce the paper's reported results. |

<a name="experimental-workflow"></a>
# 4. Experimental Workflow

The evaluated workflow uses graph-based orchestration to coordinate security analysis, retrieval, vulnerability detection, conditional repair, and structured reporting.

At a high level, the standard workflow follows this sequence:

```text
Input
  ↓
Preprocessing
  ↓
Hybrid Security Retrieval
  ↓
Vulnerability Detection
  ↓
Conditional Repair
  ↓
Structured Security Report

```

The chained-vulnerability evaluation extends this process with multi-component analysis, validated individual findings, chain reasoning, and coordinated repair planning.



<a name="environment-setup"></a>
# 5. Environment Setup

This section prepares a clean Google Colab environment for the reproduction workflow.

It performs the following tasks:

1. clone the evaluation repository;
2. install the required Python dependencies;
3. establish repository-relative paths;
4. verify the required directories and files;
5. display the active runtime environment;
6. initialize Analysis Mode without requiring API credentials.


In [ ]:
# ============================================================
# Repository Setup
# ============================================================

from pathlib import Path
import os
import subprocess
import sys

# Detect Google Colab.
try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Google Colab: {IN_COLAB}")

REPO_NAME = "llm-security-framework-evaluation"
REPO_URL = "https://github.com/lizethcampos03/llm-security-framework-evaluation.git"

if IN_COLAB:
    REPO_DIR = Path("/content") / REPO_NAME
else:
    # When run locally, locate the repository root instead of assuming
    # that the notebook was launched from the root directory.
    candidate = Path.cwd().resolve()
    while candidate != candidate.parent and not (candidate / ".git").exists():
        candidate = candidate.parent
    REPO_DIR = candidate if (candidate / ".git").exists() else Path.cwd().resolve()

print(f"Repository directory: {REPO_DIR}")


## Clone Repository

In [ ]:
if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        print("Repository already exists; updating from GitHub...")
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "fetch", "origin", "main"],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"],
            check=True,
        )

if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repository directory not found: {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Current working directory:\n{Path.cwd()}")


## Install Dependencies

In [ ]:
!pip install -q pandas numpy matplotlib

## Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

print("Imports completed successfully.")

## Repository Paths

In [ ]:
CONFIG_DIR = REPO_DIR / "configs"
RESULTS_DIR = REPO_DIR / "results"
PAPER_RESULTS = RESULTS_DIR / "paper_results"
TABLES_DIR = PAPER_RESULTS / "tables"
FIGURES_DIR = PAPER_RESULTS / "figures"
OUTPUT_DIR = RESULTS_DIR / "reproduced_results"

for required_dir in [CONFIG_DIR, PAPER_RESULTS, TABLES_DIR, FIGURES_DIR]:
    if not required_dir.exists():
        raise FileNotFoundError(
            f"Expected repository directory is missing: {required_dir}"
        )

print(f"Tables directory: {TABLES_DIR}")
print(f"Figures directory: {FIGURES_DIR}")


## Repository Validation

In [ ]:
required_files = {

    "Workflow Configuration":
        CONFIG_DIR / "workflow_configuration.md",

    "Calibration Results":
        TABLES_DIR / "calibration_results.csv",

    "Experiment 1 Results":
        TABLES_DIR / "experiment1_detection_results.csv",

    "Confusion Matrix":
        TABLES_DIR / "experiment1_confusion_matrix.csv",

    "Comparative Results":
        TABLES_DIR / "comparative_detection_results.csv",

    "End-to-End Results":
        TABLES_DIR / "end_to_end_workflow_results.csv",

    "Chain Scenarios":
        TABLES_DIR / "chained_vulnerability_scenarios.csv",

    "Chain Results":
        TABLES_DIR / "chained_vulnerability_results.csv",

    "Runtime Cost Summary":
        TABLES_DIR / "runtime_cost_summary.csv",
}

validation = []

for name, path in required_files.items():

    validation.append({
        "Artifact": name,
        "Exists": path.exists()
    })

validation_df = pd.DataFrame(validation)

display(validation_df)

## Artifact Display Utilities

The following helper functions load and display archived repository
artifacts without altering their contents.

In [ ]:
from IPython.display import Markdown, Image, display
import json


def load_reported_table(filename, required_columns=None):
    """Load an archived reported-result CSV and normalize known legacy headers."""
    path = TABLES_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Table not found: {path}")

    dataframe = pd.read_csv(path)

    # Pandas renames duplicate CSV headers with a .1 suffix. The prior
    # end-to-end table briefly used gpt4_workflow twice; the second column
    # contains the GPT-5.5 workflow values.
    legacy_renames = {
        "gpt4_workflow.1": "gpt55_workflow",
        "gpt5.5_workflow": "gpt55_workflow",
        "gpt5_5_workflow": "gpt55_workflow",
    }
    dataframe = dataframe.rename(
        columns={
            old: new
            for old, new in legacy_renames.items()
            if old in dataframe.columns
        }
    )

    if required_columns:
        missing = [
            column
            for column in required_columns
            if column not in dataframe.columns
        ]

        if missing:
            raise ValueError(
                f"{filename} is missing columns: {missing}. "
                f"Available columns: {list(dataframe.columns)}"
            )

    return dataframe


def display_markdown_file(path, title=None):
    """Display a Markdown artifact in the notebook."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Markdown file not found: {path}")

    if title:
        display(Markdown(f"### {title}"))

    display(Markdown(path.read_text(encoding="utf-8")))


def preview_json_file(path, max_characters=4000):
    """Display a concise preview of an archived JSON output."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"JSON file not found: {path}")

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    rendered = json.dumps(data, indent=2)

    if len(rendered) > max_characters:
        rendered = rendered[:max_characters] + "\n\n... preview truncated ..."

    print(rendered)


print("Artifact display utilities are ready.")


### Workflow Figure

The architecture figure is shown below.


In [ ]:
architecture_path = FIGURES_DIR / "Overall-architecture.png"

if not architecture_path.exists():
    raise FileNotFoundError(
        f"Architecture figure not found: {architecture_path}"
    )

display(Image(filename=str(architecture_path), width=1100))
print(f"Source: {architecture_path.relative_to(REPO_DIR)}")


<a name="final-configuration-summary"></a>

# 6. Frozen Configuration

The following section presents the archived configuration of the workflow
used for the reported evaluation. The configuration was selected during
calibration and frozen before the primary benchmark experiments.

### Configuration Summary

The complete archived configuration is stored in
`configs/workflow_configuration.md`. A concise reviewer-facing summary is
displayed below.

In [ ]:
configuration_summary = pd.DataFrame(
    [
        ["Orchestration framework", "LangGraph"],
        ["Detection model", "Claude Opus"],
        ["Repair model", "GPT-5.5"],
        ["Retrieval strategy", "Full Hybrid Evidence Retrieval"],
        ["Detection approach", "Code-first, context-aware analysis"],
        ["Repair routing", "Conditional"],
        ["Primary benchmark", "138 cases: 69 vulnerable and 69 safe"],
    ],
    columns=["Component", "Archived Configuration"],
)

display(configuration_summary)

### Archived Workflow Configuration

In [ ]:
configuration_path = CONFIG_DIR / "workflow_configuration.md"

display_markdown_file(configuration_path)

<a name="benchmark-inspection"></a>

# 7. Benchmark Inspection

The primary benchmark contains one vulnerable sample and one verified
safe counterpart for each of 69 CWE families, producing 138 total cases.

This section inspects the organization of the archived benchmark. It does
not execute the samples or recalculate experimental performance.

In [ ]:
SECURITYEVAL_DIR = (
    REPO_DIR
    / "datasets"
    / "securityeval"
    / "securityeval_dataset"
)

if not SECURITYEVAL_DIR.is_dir():
    raise FileNotFoundError(
        f"SecurityEval benchmark directory not found: {SECURITYEVAL_DIR}"
    )

cwe_directories = sorted(
    path
    for path in SECURITYEVAL_DIR.iterdir()
    if path.is_dir() and path.name.startswith("CWE-")
)

benchmark_summary = pd.DataFrame(
    [
        ["CWE families", len(cwe_directories)],
        ["Vulnerable samples", len(cwe_directories)],
        ["Safe counterparts", len(cwe_directories)],
        ["Total benchmark cases", len(cwe_directories) * 2],
    ],
    columns=["Artifact", "Count"],
)

display(benchmark_summary)


In [ ]:
representative_cwes = cwe_directories[:5]

sample_inventory = []

for cwe_dir in representative_cwes:
    sample_inventory.append(
        {
            "CWE": cwe_dir.name,
            "Vulnerable Sample": (
                cwe_dir / "vulnerable_securityeval_sample.py"
            ).exists(),
            "Safe Counterpart": (
                cwe_dir / "safe_verified_counterpart.py"
            ).exists(),
            "Context Profile": (
                cwe_dir / "context_profile.json"
            ).exists(),
            "Assignment Metadata": (
                cwe_dir / "assignment.json"
            ).exists(),
        }
    )

display(pd.DataFrame(sample_inventory))

### Representative Benchmark Pair

The following code samples are shown only to illustrate the structure of
the archived benchmark.

In [ ]:
example_directory = cwe_directories[0]

vulnerable_path = (
    example_directory / "vulnerable_securityeval_sample.py"
)
safe_path = (
    example_directory / "safe_verified_counterpart.py"
)

print(f"Example CWE: {example_directory.name}")

print("\n--- Vulnerable sample ---\n")
print(vulnerable_path.read_text(encoding="utf-8"))

print("\n--- Verified safe counterpart ---\n")
print(safe_path.read_text(encoding="utf-8"))

<a name="prompts-and-scripts"></a>

# 8. Prompts and Experiment Scripts

The public repository includes the prompts and experiment scripts used
during the evaluation.

The scripts document the experimental procedures but are not executed by
this notebook.

In [ ]:
PROMPTS_DIR = REPO_DIR / "prompts"

prompt_files = sorted(PROMPTS_DIR.glob("*.md"))

prompt_inventory = pd.DataFrame(
    [
        {
            "Prompt File": path.name,
            "Repository Path": str(path.relative_to(REPO_DIR)),
        }
        for path in prompt_files
        if path.name.lower() != "readme.md"
    ]
)

display(prompt_inventory)

In [ ]:
EXPERIMENTS_DIR = REPO_DIR / "experiments"

experiment_scripts = sorted(
    EXPERIMENTS_DIR.rglob("*.py")
)

script_inventory = pd.DataFrame(
    [
        {
            "Script": path.name,
            "Experiment Area": path.parent.name,
            "Repository Path": str(path.relative_to(REPO_DIR)),
        }
        for path in experiment_scripts
    ]
)

display(script_inventory)

### Representative Evaluation Prompt

In [ ]:
detector_prompt_path = PROMPTS_DIR / "detector_prompt.md"

display_markdown_file(
    detector_prompt_path,
    title="Archived Detector Prompt"
)

<a name="workflow-calibration"></a>

# 9. Reported Workflow Calibration Results

Calibration evaluated retrieval strategy, prompt design, model selection,
validation strategy, workflow structure, and repair behavior using an
18-case benchmark across nine CWE categories.

The following values are loaded directly from the archived result table
used for Table I of the paper.

In [ ]:
calibration_results_df = load_reported_table(
    "calibration_results.csv",
    required_columns=["metric", "value"],
)

display(
    calibration_results_df.style.hide(axis="index")
)

print(
    "Source:",
    (
        TABLES_DIR / "calibration_results.csv"
    ).relative_to(REPO_DIR),
)

<a name="experiment-1-detection-and-comparative-analysis"></a>

# 10. Reported Detection and Comparative Results

The following archived tables summarize the reported performance of the
frozen workflow on the 138-case benchmark and its comparison with the GPT-4 secure code agent,
CodeQL, and Bandit.

The notebook displays the reported values without independently
recalculating the experimental metrics.

### Table II — Aggregate Detection Results

In [ ]:
detection_results_df = load_reported_table(
    "experiment1_detection_results.csv",
    required_columns=["metric", "value"],
)

display(detection_results_df.style.hide(axis="index"))

### Table III — Confusion Matrix

In [ ]:
confusion_matrix_df = load_reported_table(
    "experiment1_confusion_matrix.csv",
    required_columns=["metric", "count", "description"],
)

display(confusion_matrix_df.style.hide(axis="index"))

### Table IV — Comparative Detection Performance

In [ ]:
comparative_results_df = load_reported_table(
    "comparative_detection_results.csv",
    required_columns=[
        "method",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "vulnerable_cases_detected",
    ],
)

display(comparative_results_df.style.hide(axis="index"))

<a name="experiment-2-end-to-end-secure-remediation"></a>

# 11. Reported End-to-End Remediation Results

The remediation evaluation examined the vulnerable samples that were
successfully detected by the End-to-End Workflow under the different configurations.

The following table is the archived result used for Table V of the paper.

In [ ]:
remediation_results_df = load_reported_table(
    "end_to_end_workflow_results.csv",
    required_columns=[
        "metric",
        "baseline",
        "baseline",
        "proposed_framework",
    ],
)

display(remediation_results_df.style.hide(axis="index"))


### Reported Remediation Metrics for the End-to-End Workflow using the Langgraph framework for detection

- Original vulnerable cases: **69**
- Detected and repair eligible: **66**
- Reported successful repairs: **66**
- Reported repair success among eligible cases: **100.00%**
- Reported final secure-output rate: **95.65%**

These values reflect the archived evaluation and structured manual review
described in the paper.

<a name="experiment-3-chained-vulnerability-evaluation"></a>

# 12. Reported Chained-Vulnerability Results

This preliminary evaluation considered four representative
multi-component scenarios.

The archived results report component-level detection and candidate-chain
identification for all four scenarios, with complete chain confirmation
and repair planning for three.

### Table VI — Chained-Vulnerability Scenario Overview

In [ ]:
chain_scenarios_df = load_reported_table(
    "chained_vulnerability_scenarios.csv",
    required_columns=["scenario", "focus"],
)

display(chain_scenarios_df.style.hide(axis="index"))

### Table VII — Chained-Vulnerability Evaluation Results

In [ ]:
chain_results_df = load_reported_table(
    "chained_vulnerability_results.csv"
)

display(chain_results_df.style.hide(axis="index"))

> **Interpretation limitation:** This four-scenario evaluation is a
> preliminary assessment of evidence-supported attack-path reasoning. It
> is not a generalizable attack-chain accuracy benchmark and does not
> constitute dynamic exploit validation.

<a name="runtime-and-cost-analysis"></a>

# 13. Reported Runtime and Cost

Runtime measurements were collected from LangSmith for the final workflow.

The API costs shown for the 138-case benchmark were proportionally
derived from a previously measured 370-case batch because the provider
dashboards aggregated multiple executions during the same billing period.

The complete provenance and calculation are documented in the archived
runtime and cost report.

Table VIII - Runtime and Cost Summary

In [ ]:
runtime_cost_df = load_reported_table(
    "runtime_cost_summary.csv",
    required_columns=["metric", "value"],
)

display(runtime_cost_df.style.hide(axis="index"))

### Runtime and Cost Provenance

In [ ]:
runtime_report_path = (
    PAPER_RESULTS
    / "outputs"
    / "runtime_cost"
    / "runtime_cost_report.md"
)

display_markdown_file(runtime_report_path)

<a name="paper-table-inventory"></a>

# 14. Paper Table Inventory

The following inventory maps each table in the paper to its archived
repository source.

In [ ]:
table_inventory = pd.DataFrame(
    [
        ["Table I", "Calibration Results", "calibration_results.csv"],
        [
            "Table II",
            "Aggregate Detection Results",
            "experiment1_detection_results.csv",
        ],
        [
            "Table III",
            "Confusion Matrix",
            "experiment1_confusion_matrix.csv",
        ],
        [
            "Table IV",
            "Comparative Detection Performance",
            "comparative_detection_results.csv",
        ],
        [
            "Table V",
            "End-to-End Remediation",
            "end_to_end_workflow_results.csv",
        ],
        [
            "Table VI",
            "Chain Scenario Overview",
            "chained_vulnerability_scenarios.csv",
        ],
        [
            "Table VII",
            "Chain Evaluation Results",
            "chained_vulnerability_results.csv",
        ],
        [
            "Table VIII",
            "Runtime and Cost Summary",
            "runtime_cost_summary.csv",
        ],
    ],
    columns=["Paper Table", "Description", "Archived Source"],
)

table_inventory["Available"] = table_inventory[
    "Archived Source"
].apply(
    lambda filename: (TABLES_DIR / filename).exists()
)

display(table_inventory.style.hide(axis="index"))

# Architecture and Supporting Figures

The only paper figure is the archived workflow architecture displayed
earlier in the notebook.


<a name="supporting-artifacts"></a>

# 15. Supporting Logs and Outputs

This section lists the archived logs and representative outputs supporting
the reported evaluation. Large files are inventoried rather than fully
printed.

In [ ]:
LOGS_DIR = PAPER_RESULTS / "experiment_logs"

log_files = sorted(
    path
    for path in LOGS_DIR.rglob("*")
    if path.is_file()
)

log_inventory = pd.DataFrame(
    [
        {
            "File": path.name,
            "Repository Path": str(path.relative_to(REPO_DIR)),
            "Size (KB)": round(path.stat().st_size / 1024, 2),
        }
        for path in log_files
    ]
)

display(log_inventory)

In [ ]:
ARCHIVED_OUTPUTS_DIR = PAPER_RESULTS / "outputs"

output_files = sorted(
    path
    for path in ARCHIVED_OUTPUTS_DIR.rglob("*")
    if path.is_file()
)

output_inventory = pd.DataFrame(
    [
        {
            "File": path.name,
            "Output Area": path.parent.name,
            "Repository Path": str(path.relative_to(REPO_DIR)),
            "Size (KB)": round(path.stat().st_size / 1024, 2),
        }
        for path in output_files
    ]
)

display(output_inventory)

### Representative Archived Output

In [ ]:
chain_summary_path = (
    ARCHIVED_OUTPUTS_DIR
    / "experiment_3"
    / "run_summary.json"
)

preview_json_file(chain_summary_path)

# Artifact Interpretation Notes

The repository preserves the experimental materials and reported results
used to prepare the paper.

Its purpose is to make the published evaluation transparent, organized,
and readily inspectable.

<a name="artifact-summary"></a>

# 16. Artifact Summary

The following automated checklist confirms the availability of the
repository materials referenced throughout the notebook.

In [ ]:
artifact_checks = {
    "Repository cloned": REPO_DIR.exists(),
    "Frozen configuration available": (
        CONFIG_DIR / "workflow_configuration.md"
    ).exists(),
    "Benchmark directory available": SECURITYEVAL_DIR.exists(),
    "Prompt files available": len(prompt_files) > 0,
    "Experiment scripts available": len(experiment_scripts) > 0,
    "Tables I-VIII available": table_inventory["Available"].all(),
    "Experiment logs available": len(log_files) > 0,
    "Representative outputs available": len(output_files) > 0,
    "Architecture figure available": architecture_path.exists(),
    "Runtime and cost report available": runtime_report_path.exists(),
}

artifact_summary_df = pd.DataFrame(
    [
        {
            "Artifact Check": name,
            "Available": available,
            "Status": "Ready" if available else "Missing",
        }
        for name, available in artifact_checks.items()
    ]
)

display(artifact_summary_df)

if not artifact_summary_df["Available"].all():
    raise FileNotFoundError(
        "One or more evaluation artifacts are missing."
    )

print(
    "\nEvaluation artifact inspection completed successfully.\n"
    "All reported tables and core supporting artifacts were located."
)

<a name="references"></a>

# 17. References

Primary resources associated with this notebook include:

- the accompanying conference paper;
- the public evaluation repository;
- SecurityEval;
- MITRE CWE;
- MITRE CAPEC;
- CodeQL;
- Bandit;
- LangGraph;
- LangSmith;
- Anthropic;
- OpenAI.

Complete academic citations are provided in the accompanying paper.

---

## End of Evaluation Artifact Notebook

This notebook provides a structured interface for inspecting the archived
configuration, benchmark, prompts, experiment scripts, logs, outputs, and
results reported in the accompanying paper.